In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm, trange
from sklearn.decomposition import PCA

In [11]:
# 1. Load data
train_data = torch.load('data/bert_embeddings_lang.pt', map_location='cpu')

# lang_mask = [lang == 'deu_Latn' for lang in train_data['lang']]
X = train_data['embedding']#[lang_mask]
y = train_data['label']#[lang_mask]

# 3. Stratified Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

# Ordinal targets: for a label k in {0, 1, 2, 3, 4},
# use four binary thresholds [y > 0, y > 1, y > 2, y > 3].
def to_ordinal_targets(labels, num_thresholds=4):
    return torch.stack([(labels > threshold).float() for threshold in range(num_thresholds)], dim=1)

y_train = to_ordinal_targets(y_train)
y_val = to_ordinal_targets(y_val)

# print("PCAing the data for whitening...")
# whitener = PCA(n_components=None, whiten=True)

# # Fit ONLY on training data, then transform both
# X_train = torch.tensor(whitener.fit_transform(X_train), dtype=torch.float32)
# X_val = torch.tensor(whitener.transform(X_val), dtype=torch.float32)
# print("Done")

In [12]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

In [13]:
# 4. MLP Architecture
class SentimentMLP(nn.Module):
    def __init__(self, input_dim=768, num_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = SentimentMLP(num_classes=4)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# 5. Training Loop with Nested Progress Bars
epochs = 20
outer_bar = tqdm(range(epochs), desc="Overall Progress")

for epoch in outer_bar:
    model.train()
    epoch_loss = 0
    
    # Inner bar for batches
    inner_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)
    
    for batch_X, batch_y in inner_bar:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        inner_bar.set_postfix(loss=f"{loss.item():.4f}")
    
    # 6. Validation & Custom Scoring
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).long()
            
            all_preds.append(preds.sum(dim=1))
            all_targets.append(batch_y.sum(dim=1).long())

        # Concatenate everything back into single vectors
        all_preds = torch.cat(all_preds).cpu().numpy()
        all_targets = torch.cat(all_targets).cpu().numpy()

        # 2. Calculate Metrics on the gathered results
        mae = mean_absolute_error(all_targets, all_preds)
        score_criterion = 1 - (mae / 4)
        acc = (all_preds == all_targets).mean()
        
    # Update outer bar with the metrics that matter
    outer_bar.set_postfix(
        score=f"{score_criterion:.4f}", 
        mae=f"{mae:.4f}", 
        acc=f"{acc:.2f}"
    )

print("\nTraining Complete.")

Overall Progress:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 2:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 3:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 4:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 5:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 6:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 7:   0%|          | 0/3544 [00:00<?, ?it/s]

Epoch 8:   0%|          | 0/3544 [00:00<?, ?it/s]

KeyboardInterrupt: 